# Q1 Validation Notebook

This notebook validates the calculations and chart logic currently used in the Streamlit `Product Analysis` page.

It reproduces:
- Research API acquisition and churn KPI
- Retention comparison (first action = query vs research)
- Pareto concentration of research traffic
- Average response time by model (`mini` vs `pro`)


## 1) Load Data

**What is calculated here:**
- Load all four required datasets (`hourly_usage`, `Infrastructure_costs`, `research_requests`, `users`).
- Normalize column names to lowercase for consistent downstream logic.

**How it is calculated:**
- Try reading from `data.zip` first.
- If zip is missing/invalid, fallback to local CSV files.

**Expected output:**
- Printed shape for each loaded table.

In [9]:
import zipfile
from pathlib import Path

import pandas as pd
import plotly.express as px

APP_DIR = Path.cwd()
ZIP_PATH = APP_DIR / "data.zip"

required_files = [
    "hourly_usage.csv",
    "infrastructure_costs.csv",
    "research_requests.csv",
    "users.csv",
]

frames = {}
if ZIP_PATH.is_file():
    try:
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            members = set(zf.namelist())
            missing = [name for name in required_files if name not in members]
            if not missing:
                for name in required_files:
                    with zf.open(name) as f:
                        frames[name] = pd.read_csv(f)
    except zipfile.BadZipFile:
        frames = {}

if not frames:
    for name in required_files:
        for candidate in (APP_DIR / name, APP_DIR.parent / name):
            if candidate.is_file():
                frames[name] = pd.read_csv(candidate)
                break
        else:
            raise FileNotFoundError(f"Could not load '{name}' from zip or CSV fallback.")

hourly_usage = frames["hourly_usage.csv"].copy()
Infrastructure_costs = frames["infrastructure_costs.csv"].copy()
research_requests = frames["research_requests.csv"].copy()
users = frames["users.csv"].copy()

for df in (hourly_usage, Infrastructure_costs, research_requests, users):
    df.columns = df.columns.str.lower()

print("Loaded:")
print("hourly_usage", hourly_usage.shape)
print("Infrastructure_costs", Infrastructure_costs.shape)
print("research_requests", research_requests.shape)
print("users", users.shape)


Loaded:
hourly_usage (972146, 7)
Infrastructure_costs (3611, 17)
research_requests (116174, 17)
users (16324, 5)


## 2) Build User Lifecycle Table

**Step 1 goal:**
- Count users who joined on/after `2025-11-01` from `users.csv`.

**How it is calculated:**
- Parse `created_at` as UTC datetime.
- Keep valid `user_id` + `created_at` rows.
- Filter `created_at >= 2025-11-01`.
- Count unique `user_id`.

**Expected output:**
- Printed total users joined since Nov 1, 2025.
- Small preview table of filtered users.

In [19]:
# Build User Lifecycle Table — Step 1
# Calculate number of users who joined on/after Nov 1, 2025 from users.csv

users_clean = users[["user_id", "created_at"]].copy()
users_clean["created_at"] = pd.to_datetime(users_clean["created_at"], errors="coerce", utc=True)
users_clean["user_id"] = pd.to_numeric(users_clean["user_id"], errors="coerce")
users_clean = users_clean.dropna(subset=["user_id", "created_at"]).copy()
users_clean["user_id"] = users_clean["user_id"].astype(int)

new_users = users_clean.loc[
    users_clean["created_at"] >= pd.Timestamp("2025-11-01", tz="UTC"),
    ["user_id", "created_at"],
].drop_duplicates(subset=["user_id"])

print(f"Users joined since 2025-11-01: {new_users['user_id'].nunique():,}")
print("Preview of new users:")
print(new_users.head())


Users joined since 2025-11-01: 12,895
Preview of new users:
   user_id                created_at
0        1 2025-12-10 10:09:27+00:00
3        4 2026-02-19 01:11:57+00:00
4        5 2026-02-24 05:42:51+00:00
8        9 2026-01-14 05:09:52+00:00
9       10 2025-11-12 23:51:12+00:00


## 3) KPI Validation: Acquisition and Churn

**What is calculated here:**
- New users created since the beginning of November 2025.
- Acquisition % = share of those users whose first action is Research.
- Churn % among Research-first users.

**How it is calculated:**
- New-user start date is fixed to `2025-11-01`.
- First action derived from lifecycle table.
- Churn = `100 - retention` where retention means active at least 30 days after first action.

**Expected output:**
- Printed November start date, user counts, acquisition %, churn %.

In [18]:
# KPI validation: acquisition + churn (new users since Nov 1, 2025)
nov_start = pd.Timestamp("2025-11-01", tz="UTC")

new_users = users_clean.loc[users_clean["created_at"] >= nov_start, ["user_id"]].drop_duplicates()
new_lifecycle = new_users.merge(lifecycle, on="user_id", how="inner")

if len(new_lifecycle) == 0:
    acquisition_pct = 0.0
    churn_pct = 0.0
    research_first_users_count = 0
else:
    research_first = new_lifecycle["first_source"].eq("research")
    acquisition_pct = 100.0 * research_first.mean()
    research_first_users = new_lifecycle.loc[research_first]
    research_first_users_count = len(research_first_users)
    if research_first_users_count == 0:
        churn_pct = 0.0
    else:
        retention_pct = 100.0 * research_first_users["retained_30d"].mean()
        churn_pct = 100.0 - retention_pct

print(f"New-user window starts at: {nov_start}")
print(f"New users since Nov 1: {len(new_users):,}")
print(f"New users with any valid activity: {len(new_lifecycle):,}")
print(f"Users acquired via Research API first action: {research_first_users_count:,}")
print(f"Research API Acquisition: {acquisition_pct:.2f}%")
print(f"Churn among Research-first users: {churn_pct:.2f}%")


New-user window starts at: 2025-11-01 00:00:00+00:00
New users since Nov 1: 12,895
New users with any valid activity: 12,006
Users acquired via Research API first action: 1,468
Research API Acquisition: 12.23%
Churn among Research-first users: 88.83%


## 4) Chart Prototype: Retention by First Action

**What is calculated here:**
- Retention rate split by first action (`Query` vs `Research`).

**How it is calculated:**
- Group users by `first_source` from lifecycle.
- Compute mean of `retained_30d` and convert to percentage.

**Expected output:**
- Printed retention summary table.
- Plotly bar chart.

In [12]:
# Chart prototype 1: Retention comparison
retention_df = lifecycle.copy()
retention_df["segment"] = retention_df["first_source"].map({
    "query": "First Action = Query",
    "research": "First Action = Research",
})
retention_df = (
    retention_df.groupby("segment", as_index=False)["retained_30d"]
    .mean()
    .rename(columns={"retained_30d": "retention_rate"})
)
retention_df["retention_rate"] = retention_df["retention_rate"] * 100.0

print("Retention table (%):")
print(retention_df)

fig_retention = px.bar(
    retention_df,
    x="segment",
    y="retention_rate",
    title="Retention Rate by First Action",
    labels={"segment": "First Action", "retention_rate": "Retention Rate (%)"},
    text=retention_df["retention_rate"].map(lambda x: f"{x:.1f}%"),
)
fig_retention.update_layout(template="simple_white")
fig_retention.show()


Retention table (%):
                   segment  retention_rate
0     First Action = Query       38.792236
1  First Action = Research        8.036454


## 5) Chart Prototype: Average Response Time (Mini vs Pro)

**What is calculated here:**
- Mean `response_time_seconds` for `mini` and `pro` models.

**How it is calculated:**
- Filter `research_requests` to `model in [mini, pro]`.
- Drop invalid response times.
- Group by model and calculate mean.

**Expected output:**
- Printed model latency table.
- Plotly horizontal bar chart.

In [13]:
# Chart prototype 2: Average response time by model (mini vs pro)
latency_df = research_requests.copy()
latency_df["model"] = latency_df["model"].astype(str).str.lower().str.strip()
latency_df["response_time_seconds"] = pd.to_numeric(latency_df["response_time_seconds"], errors="coerce")
latency_df = (
    latency_df[latency_df["model"].isin(["mini", "pro"])]
    .dropna(subset=["response_time_seconds"])
    .groupby("model", as_index=False)["response_time_seconds"]
    .mean()
    .rename(columns={"response_time_seconds": "avg_response_time_seconds"})
)

print("Average response time by model:")
print(latency_df)

fig_latency = px.bar(
    latency_df,
    x="avg_response_time_seconds",
    y="model",
    orientation="h",
    title="Average Response Time by Model (Mini vs Pro)",
    labels={
        "avg_response_time_seconds": "Average Response Time (seconds)",
        "model": "Model",
    },
    text=latency_df["avg_response_time_seconds"].map(lambda x: f"{x:.2f}s"),
)
fig_latency.update_layout(template="simple_white")
fig_latency.show()


Average response time by model:
  model  avg_response_time_seconds
0  mini                  38.267307
1   pro                 306.939992


## 6) Chart Prototype: Pareto Traffic Concentration

**What is calculated here:**
- Cumulative % of users vs cumulative % of research requests.
- Share of requests generated by top 5% of users.

**How it is calculated:**
- Count requests per user and sort descending.
- Compute cumulative percentages for users and requests.
- Draw reference lines at 5% users and corresponding request share.

**Expected output:**
- Printed top-5% request share.
- Plotly Pareto line chart.

In [14]:
# Chart prototype 3: Pareto concentration of research traffic
pareto_raw = research_requests.copy()
pareto_raw["user_id"] = pd.to_numeric(pareto_raw["user_id"], errors="coerce")
pareto_raw = pareto_raw.dropna(subset=["user_id"]).copy()
pareto_raw["user_id"] = pareto_raw["user_id"].astype(int)

counts = pareto_raw.groupby("user_id").size().sort_values(ascending=False)
pareto = pd.DataFrame({"requests": counts.values})
pareto["cum_requests_pct"] = 100.0 * pareto["requests"].cumsum() / pareto["requests"].sum()
pareto["cum_users_pct"] = 100.0 * (pareto.index + 1) / len(pareto)
pareto = pd.concat(
    [
        pd.DataFrame({"cum_users_pct": [0.0], "cum_requests_pct": [0.0]}),
        pareto[["cum_users_pct", "cum_requests_pct"]],
    ],
    ignore_index=True,
)

fig_pareto = px.line(
    pareto,
    x="cum_users_pct",
    y="cum_requests_pct",
    title="Research API Traffic Concentration (Pareto Curve)",
    labels={
        "cum_users_pct": "Cumulative % of Users",
        "cum_requests_pct": "Cumulative % of Total Requests",
    },
)
y_at_5 = float(
    pareto.loc[pareto["cum_users_pct"] >= 5.0, "cum_requests_pct"].head(1).fillna(0.0).iloc[0]
)
fig_pareto.add_vline(x=5.0, line_dash="dash", line_color="gray")
fig_pareto.add_hline(y=y_at_5, line_dash="dash", line_color="gray")
fig_pareto.update_layout(template="simple_white")
fig_pareto.show()

print(f"Users in Pareto analysis: {len(counts):,}")
print(f"At 5% of users, cumulative request share is {y_at_5:.2f}%")


Users in Pareto analysis: 16,333
At 5% of users, cumulative request share is 59.94%
